# 🔬 BioHub Cell Tracking — Kaggle Submission Notebook

**Competition:** [Biohub - Cell Tracking During Development](https://www.kaggle.com/competitions/biohub-cell-tracking-during-development)

**Score formula:** `edge_jaccard + 0.1 × division_jaccard`

### Pipeline overview
1. **Segmentation** — Cellpose 3D (cyto3 model) with blob-detector fallback
2. **Post-processing** — size filter (50–50k voxels) + relabelling
3. **Tracking** — Hungarian linker (max 10 µm, volume-cost enabled) + gap-2 bridging
4. **Division detection** — orphan-pair volume-consistency classifier
5. **Submission** — 10-column CSV (nodes + edges)

In [18]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
import sys, subprocess

print(f'Python {sys.version}')

# Uncomment any missing packages:
# subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
#                 'cellpose>=3.0.0', 'scikit-image>=0.21', 'scipy',
#                 'zarr>=3.0', 'tqdm'], check=True)

Python 3.12.3 (tags/v3.12.3:f6650f9, Apr  9 2024, 14:05:25) [MSC v.1938 64 bit (AMD64)]


In [ ]:
# ── Cell 2: Local path registration (No setup.py/pip install needed) ─────────
import pathlib, sys, importlib

# 1. Tenter de localiser le dépôt cloné ou téléversé
REPO_ROOT = pathlib.Path('/kaggle/working/biohub-cell-tracking')
if not REPO_ROOT.exists():
    REPO_ROOT = pathlib.Path('/kaggle/input/biohub-cell-tracking-code')
if not REPO_ROOT.exists():
    # En local sur votre PC
    REPO_ROOT = pathlib.Path('..').resolve()

print(f"Repo root path: {REPO_ROOT}")

# 2. Enregistrer le dossier 'src' dans sys.path de ce noyau Jupyter
src_path = str(REPO_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
    print(f"Added to sys.path: {src_path}")
else:
    print("sys.path already registered")

# 3. Rafraîchir les caches d'importation de Python
importlib.invalidate_caches()

# 4. Vérifier si le package est maintenant correctement lié et importable
spec = importlib.util.find_spec('biohub_tracking')
if spec:
    print(f"✅ biohub_tracking successfully found at: {spec.origin}")
else:
    raise ImportError(
        f"biohub_tracking not found!\n"
        f"Please verify that the directory '{src_path}' exists and contains "
        f"the package source files."
    )


Repo root      : C:\Users\nidha\Desktop\github reposistory\biohub-cell-tracking
setup.py found : True
pip install -e  : done
sys.path update : added C:\Users\nidha\Desktop\github reposistory\biohub-cell-tracking\src
✅ biohub_tracking: C:\Users\nidha\Desktop\github reposistory\biohub-cell-tracking\src\biohub_tracking\__init__.py


In [20]:
# ── Cell 3: Configuration (auto-detects Kaggle vs local) ──────────────────────
import logging

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(message)s',
    handlers=[logging.StreamHandler()],
)

# Auto-detect environment
ON_KAGGLE = pathlib.Path('/kaggle').exists()

if ON_KAGGLE:
    TEST_DIR = pathlib.Path('/kaggle/input/biohub-cell-tracking-during-development/test')
    OUTPUT   = pathlib.Path('/kaggle/working/submission.csv')
else:
    # Local: download a few samples and place them in REPO_ROOT/data/test/
    #   kaggle competitions download -c biohub-cell-tracking-during-development -f <file>
    TEST_DIR = REPO_ROOT / 'data' / 'test'
    OUTPUT   = REPO_ROOT / 'submission.csv'

print(f'Environment : {"Kaggle" if ON_KAGGLE else "Local"}')
print(f'Test dir    : {TEST_DIR}')
print(f'Output CSV  : {OUTPUT}')

zarr_files = sorted(TEST_DIR.glob('*.zarr')) if TEST_DIR.exists() else []
print(f'Samples found: {len(zarr_files)}')
for p in zarr_files[:5]:
    print(f'  {p.name}')
if len(zarr_files) > 5:
    print(f'  ... and {len(zarr_files) - 5} more.')
if not zarr_files:
    print('⚠️  No .zarr files found.')
    if not ON_KAGGLE:
        print(f'   Place test data in: {TEST_DIR}')
        print('   Or download one sample with:')
        print('   kaggle competitions download -c biohub-cell-tracking-during-development -f <filename>')

Environment : Local
Test dir    : C:\Users\nidha\Desktop\github reposistory\biohub-cell-tracking\data\test
Output CSV  : C:\Users\nidha\Desktop\github reposistory\biohub-cell-tracking\submission.csv
Samples found: 0
⚠️  No .zarr files found.
   Place test data in: C:\Users\nidha\Desktop\github reposistory\biohub-cell-tracking\data\test
   Or download one sample with:
   kaggle competitions download -c biohub-cell-tracking-during-development -f <filename>


In [21]:
# ── Cell 4: Inspect one sample ────────────────────────────────────────────────
if zarr_files:
    import zarr, numpy as np
    sample = zarr_files[0]
    root   = zarr.open(str(sample), mode='r')
    arr    = root['0'] if '0' in root else root
    print(f'Sample : {sample.name}')
    print(f'Shape  : {arr.shape}  (T, Z, Y, X)')
    print(f'dtype  : {arr.dtype}')
    print(f'Chunks : {arr.chunks}')
else:
    print('No samples available — skipping inspection.')

No samples available — skipping inspection.


In [22]:
# ── Cell 5: Run full pipeline & generate submission ───────────────────────────
# Imports from the installed biohub_tracking package (registered in Cell 2).

from biohub_tracking.data.zarr_loader import iter_frames
from biohub_tracking.evaluation.submission_builder import SubmissionBuilder
from biohub_tracking.segmentation.segmenter import CellSegmenter
from biohub_tracking.tracking.division_classifier import DivisionClassifier
from biohub_tracking.tracking.linker import HungarianLinker

# ── Constants (match competition voxel calibration) ──
VOXEL_SIZE_UM    = (1.625, 0.40625, 0.40625)   # Z, Y, X in µm
ANISOTROPY       = VOXEL_SIZE_UM[0] / VOXEL_SIZE_UM[1]   # ≈ 4.0
LINK_MAX_DIST_UM = 10.0    # linker cutoff (metric threshold is 7 µm)


def make_segmenter():
    return CellSegmenter(
        method='cellpose',       # falls back to blob if Cellpose unavailable
        diameter=12.0,
        do_3D=True,
        anisotropy=ANISOTROPY,
        flow_threshold=0.4,
        cellprob_threshold=0.0,
        min_size=50,
        max_volume=50_000,
        channels=(0, 0),
        voxel_size_um=VOXEL_SIZE_UM,
        model_type='cyto3',
    )


def make_linker():
    return HungarianLinker(
        max_distance=LINK_MAX_DIST_UM,
        use_volume_cost=True,
        volume_weight=0.3,
    )


def process_sample(sample_path, segmenter):
    """Segment + link + detect divisions for one .zarr sample."""
    all_cells = {}
    for frame_index, image in iter_frames(sample_path):
        _, cells = segmenter.segment_frame(image, frame_index)
        all_cells[frame_index] = cells

    linker = make_linker()
    links = []
    linked_sources = {f: set() for f in all_cells}
    linked_targets = {f: set() for f in all_cells}
    linked_ids = {}
    sorted_frames = sorted(all_cells)

    # Primary: consecutive-frame linking
    for frame in sorted_frames[:-1]:
        fl = linker.link(all_cells[frame], all_cells[frame + 1])
        for src, tgt, conf in fl:
            links.append((frame, src, tgt, conf))
            linked_sources[frame].add(src)
            linked_targets[frame + 1].add(tgt)
        linked_ids[frame] = [(s, t) for s, t, _ in fl]

    # Gap-2 bridging: reconnect cells that disappear for 1 frame
    for frame in sorted_frames[:-2]:
        t2 = frame + 2
        if t2 not in all_cells:
            continue
        orphan_src = [c for c in all_cells[frame] if c.id not in linked_sources[frame]]
        orphan_tgt = [c for c in all_cells[t2]    if c.id not in linked_targets[t2]]
        if not orphan_src or not orphan_tgt:
            continue
        for src, tgt, conf in linker.link(orphan_src, orphan_tgt):
            links.append((frame, src, tgt, conf * 0.9))
            linked_ids.setdefault(frame, []).append((src, tgt))
            linked_sources[frame].add(src)
            linked_targets[t2].add(tgt)

    divisions = DivisionClassifier(max_distance_um=10.0).detect(all_cells, linked_ids)
    return all_cells, links, divisions


def generate_submission(test_dir, output_path):
    seg     = make_segmenter()
    builder = SubmissionBuilder()
    rows    = []
    samples = sorted(pathlib.Path(test_dir).glob('*.zarr'))
    if not samples:
        raise FileNotFoundError(f'No .zarr samples in {test_dir}')
    for sp in samples:
        logging.info('Processing %s', sp.name)
        all_cells, links, divisions = process_sample(sp, seg)
        rows.extend(builder.build_rows(sp.stem, all_cells, links, divisions))
    builder.write(rows, output_path)
    logging.info('Wrote %d rows to %s', len(rows), output_path)
    return len(rows)


# ── Run ──
if zarr_files:
    n = generate_submission(TEST_DIR, OUTPUT)
    print(f'\n✅ Done — {n:,} rows written to {OUTPUT}')
else:
    print('⚠️  Skipped: no test samples found (see Cell 3 for instructions).')

ModuleNotFoundError: No module named 'numpy'

In [ ]:
# ── Cell 6: Validate and preview submission ───────────────────────────────────
import pandas as pd

df = pd.read_csv(OUTPUT)

print('=' * 60)
print(f'Total rows   : {len(df):,}')
print(f'Node rows    : {(df.row_type == "node").sum():,}')
print(f'Edge rows    : {(df.row_type == "edge").sum():,}')
print(f'Datasets     : {df.dataset.nunique()}')
print(f'Columns      : {list(df.columns)}')
print('=' * 60)

# Schema checks
assert list(df.columns) == ['id', 'dataset', 'row_type', 'node_id', 't', 'z', 'y', 'x', 'source_id', 'target_id'], \
    'Column names do not match expected schema!'
assert df.row_type.isin(['node', 'edge']).all(), 'Unexpected row_type values!'

nodes = df[df.row_type == 'node']
edges = df[df.row_type == 'edge']

assert (nodes.node_id > 0).all(), 'Node rows must have positive node_id'
assert (nodes.t >= 0).all(), 'Node rows must have t >= 0'

all_node_ids = set(nodes.node_id)
assert edges.source_id.isin(all_node_ids).all(), 'Edge source_id references unknown node'
assert edges.target_id.isin(all_node_ids).all(), 'Edge target_id references unknown node'

print('\n✅ All validation checks passed!')
print('\nFirst 10 rows:')
df.head(10)

In [ ]:
# ── Cell 7: Per-dataset statistics ────────────────────────────────────────────
stats = df.groupby(['dataset', 'row_type']).size().unstack(fill_value=0)
stats['edge_ratio'] = (stats.get('edge', 0) / stats.get('node', 1)).round(2)
print(stats.to_string())

## 📤 Submitting on Kaggle

1. Open [Kaggle Competitions → Code → New Notebook](https://www.kaggle.com/competitions/biohub-cell-tracking-during-development/code)
2. **Add Input** → attach the competition dataset (87 GB mounts for free, no download)
3. Upload this repo as a **Kaggle Dataset** (zip the repo, upload → attach as `/kaggle/input/biohub-cell-tracking-code`)
4. Run all cells → `submission.csv` appears in `/kaggle/working/`
5. Click **Submit** in the Output panel

---

## 🛠️ Tuning Tips

| Parameter | Where | Effect |
|-----------|--------|--------|
| `LINK_MAX_DIST_UM` | Cell 5 | Larger → more links (fewer FN, more FP) |
| `min_size` | `make_segmenter()` | Higher → fewer debris cells |
| `diameter` | `make_segmenter()` | Match your cell size in pixels |
| `max_distance_um` | `DivisionClassifier` | Wider → finds more divisions |

```bash
# Install Cellpose for best segmentation quality:
!pip install cellpose>=3.0.0
```